# Probability, Likelihood, and Bayesian Foundations
## Advanced Computational Physics — integrated prerequisite laboratory

**Outcomes:** distinguish probabilities and densities; construct unbinned and binned likelihoods; estimate parameters and uncertainty; apply Bayes' rule; interpret a detector response; implement iterative unfolding with explicit efficiency; and recognize what is still required for full posterior inference.

Run from top to bottom with Python 3.12 and the accompanying requirements file. All data are synthetic. No original attachments or online downloads are required after installation. Fixed seeds and executed outputs are included.

### Road map
A. Distributions and histogram normalization  
B. Maximum likelihood: Poisson counts and exponential waiting times  
C. Gaussian estimation, joint parameters, and binned likelihood  
D. Coin-flip likelihood and Bayes' rule  
E. Detector response and iterative unfolding  
F. Uncertainty, forward folding, and the Bayesian-analysis bridge

In [ ]:
import os, json, platform, importlib.metadata as metadata
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import minimize, minimize_scalar, brentq, curve_fit
from scipy.special import gammaln, xlogy
from scipy.integrate import cumulative_trapezoid

plt.rcParams.update({'figure.figsize':(8,4.5), 'font.size':11,
    'axes.spines.top':False,'axes.spines.right':False,'axes.grid':True,'grid.alpha':0.2})
np.set_printoptions(precision=5,suppress=True)
print('Python',platform.python_version())
for package in ['numpy','scipy','matplotlib']:print(package,metadata.version(package))

slide_data={}
def record(name,x,**series):
    slide_data[name]={'x':np.asarray(x).tolist(),'series':{k:np.asarray(v).tolist() for k,v in series.items()}}

def show():plt.tight_layout();plt.show()

def interval_from_loglike(loglike, estimate, left, right, drop=.5):
    target=loglike(estimate)-drop
    return (brentq(lambda t:loglike(t)-target,left,estimate),
            brentq(lambda t:loglike(t)-target,estimate,right))

## A. Probability distributions and histograms
### A1. PMF, PDF, and bin probability
A probability mass function gives $P(K=k)$. A continuous density has units inverse to its variable and integrates to a probability. A density can exceed one.

For Gaussian data, the probability in a finite bin is an integral, not the PDF height at its center:
$$P(a<X<b)=\Phi((b-\mu)/\sigma)-\Phi((a-\mu)/\sigma).$$
This distinction is essential for binned likelihoods.

In [ ]:
mu_true,sigma_true=10.0,2.5
x_pdf=np.linspace(0,20,401)
k_values=np.arange(13)

print('Gaussian probability from 9 to 11:',stats.norm.cdf(11,10,2.5)-stats.norm.cdf(9,10,2.5))
print('A narrow Gaussian can have density > 1:',stats.norm.pdf(0,0,0.1))

fig,ax=plt.subplots(1,2,figsize=(11,4))
ax[0].plot(k_values,stats.poisson.pmf(k_values,2.5),'o-');ax[0].set(xlabel='Count k',ylabel='Probability mass',title='Poisson PMF')
ax[1].plot(x_pdf,stats.norm.pdf(x_pdf,10,2.5));ax[1].set(xlabel='Measurement x',ylabel='Probability density',title='Gaussian PDF')
show()

record('poisson_pmf',k_values,PMF=stats.poisson.pmf(k_values,2.5))
record('gaussian_pdf',x_pdf,PDF=stats.norm.pdf(x_pdf,10,2.5))

### A2. Counts versus `density=True`
For bins of width $\Delta x_j$, a density histogram returns $h_j=n_j/(N\Delta x_j)$. Thus $\sum h_j\Delta x_j=1$, while $\sum h_j$ need not equal one. With an explicit range, NumPy normalizes by the number **inside the range**, not the original sample size. Recovering counts with the original $N$ is wrong when observations were excluded.

Use raw counts for a count likelihood. Rebinning loses information about positions within each bin.

In [ ]:
rng_hist=np.random.default_rng(20260420)
data_hist=rng_hist.normal(mu_true,sigma_true,1000)
edges=np.linspace(5,15,11)

hist_counts,_=np.histogram(data_hist,bins=edges)
density,_=np.histogram(data_hist,bins=edges,density=True)
widths=np.diff(edges);centers=(edges[1:]+edges[:-1])/2
n_inside=hist_counts.sum()
recovered=density*n_inside*widths

print('All observations / inside histogram range:',len(data_hist),n_inside)
print('Density area:',np.sum(density*widths))
assert np.allclose(recovered,hist_counts)

plt.bar(centers,density,width=widths*0.9,alpha=0.5,label='Density histogram')
plt.plot(x_pdf,stats.norm.pdf(x_pdf,10,2.5),label='Unconditional Gaussian PDF')
plt.xlabel('x');plt.ylabel('Density');plt.legend();show()
record('histogram',centers,Density=density)

## B. Maximum likelihood in counting and timing experiments
### B1. A likelihood is a function of parameters
For conditionally independent measurements, $L(\theta)=\prod_i p(x_i\mid\theta)$ and $\log L(\theta)=\sum_i\log p(x_i\mid\theta)$. We vary parameters while holding observed data fixed. Products can underflow; log likelihoods are more stable.

For equal-exposure Poisson counts,
$$\log L(\lambda)=\sum_i[-\lambda+k_i\log\lambda-\log(k_i!)],\qquad\hat\lambda=\bar k.$$
With known exposure $\Delta t$, rate $R=\lambda/\Delta t$. The next example is for $R=5$ s$^{-1}$ and $\Delta t=0.5$ s.

In [ ]:
rng_pois=np.random.default_rng(20)
R_true,dt,n_measure=5.0,0.5,100
counts=rng_pois.poisson(R_true*dt,n_measure)

def poisson_loglike(lam,data):
    if lam<0:return -np.inf
    return float(np.sum(-lam+xlogy(data,lam)-gammaln(data+1)))

lam_mle=counts.mean()
lam_fit=minimize_scalar(lambda v:-poisson_loglike(v,counts),bounds=(0.01,10.0),method='bounded').x
ci_pois=interval_from_loglike(lambda v:poisson_loglike(v,counts),lam_mle,0.0001,10.0)
lam_grid=np.linspace(0.8,4.2,401)
delta_pois=np.array([poisson_loglike(v,counts) for v in lam_grid])-poisson_loglike(lam_mle,counts)

print('Total counts:',counts.sum(),'lambda MLE:',lam_mle,'rate MLE [1/s]:',lam_mle/dt)
print('Delta-logL = -0.5 interval for lambda:',ci_pois)
assert abs(lam_fit-lam_mle)<1e-5

plt.plot(lam_grid,delta_pois);plt.axhline(-0.5,color='gray',ls='--')
plt.ylim(-12,0.2);plt.xlabel('Expected counts per exposure');plt.ylabel('Delta log likelihood');show()
record('poisson_scan',lam_grid,Delta_logL=delta_pois)
slide_data['poisson_summary']={'mle':float(lam_mle),'rate':float(lam_mle/dt),'ci':ci_pois}

### B2. Event count and waiting time are different random variables
A constant-rate Poisson process has Poisson counts in a fixed interval and exponential waiting times. The exponential mean lifetime is $\tau=1/R$:
$$p(t\mid\tau)=\tau^{-1}e^{-t/\tau},\quad \log L(\tau)=-n\log\tau-\sum_i t_i/\tau,\quad\hat\tau=\bar t.$$
Here we separate the count PMF from the waiting-time PDF. Assume independent, uncensored waiting times, negligible dead time, and a stationary rate.

In [ ]:
rng_life=np.random.default_rng(21)
tau_true=0.2
times=rng_life.exponential(tau_true,100)
tau_mle=times.mean()

def lifetime_loglike(tau):
    return -len(times)*np.log(tau)-times.sum()/tau if tau>0 else -np.inf

tau_grid=np.linspace(0.08,0.40,401)
ci_life=interval_from_loglike(lifetime_loglike,tau_mle,0.001,2.0)

print('Lifetime MLE [s]:',tau_mle,'Rate estimate [1/s]:',1/tau_mle)
print('Likelihood interval for lifetime:',ci_life)

plt.plot(tau_grid,[lifetime_loglike(t)-lifetime_loglike(tau_mle) for t in tau_grid])
plt.axhline(-.5,color='gray',ls='--');plt.ylim(-12,0.2)
plt.xlabel('Mean lifetime tau [s]');plt.ylabel('Delta log likelihood');show()
record('lifetime',tau_grid,Delta_logL=np.array([lifetime_loglike(t) for t in tau_grid])-lifetime_loglike(tau_mle))

## C. Gaussian likelihood and uncertainty
### C1. Known measurement standard deviation
For $x_i\sim\mathcal N(\mu,\sigma^2)$ with known $\sigma$,
$$\log L(\mu)=-\frac n2\log(2\pi\sigma^2)-\frac{1}{2\sigma^2}\sum_i(x_i-\mu)^2.$$
The MLE is $\hat\mu=\bar x$, and its sampling standard deviation is $\sigma/\sqrt n$. Here the log likelihood is exactly quadratic in $\mu$. The cutoff $\Delta\log L=-1/2$ gives the central 68.27% normal confidence interval for the mean in this model.

In [ ]:
rng_gauss=np.random.default_rng(22)
x_gauss=rng_gauss.normal(10,2.5,10)
mu_mle=x_gauss.mean()

def normal_loglike(mu,sigma,data=x_gauss):
    if sigma<=0:return -np.inf
    return float(np.sum(stats.norm.logpdf(data,mu,sigma)))

mu_grid=np.linspace(6,14,401)
known_scan=np.array([normal_loglike(v,2.5) for v in mu_grid])-normal_loglike(mu_mle,2.5)
ci_normal=interval_from_loglike(lambda v:normal_loglike(v,2.5),mu_mle,0,20)
analytic_ci=mu_mle+np.array([-1,1])*2.5/np.sqrt(len(x_gauss))

print('Measurements:',x_gauss)
print('Mean MLE:',mu_mle,'Known-sigma interval:',ci_normal)
assert np.allclose(ci_normal,analytic_ci)

plt.plot(mu_grid,known_scan);plt.axhline(-0.5,color='gray',ls='--')
plt.ylim(-12,0.2);plt.xlabel('Gaussian mean mu');plt.ylabel('Delta log likelihood');show()
record('gaussian_scan',mu_grid,Delta_logL=known_scan)
slide_data['normal_summary']={'mle':float(mu_mle),'ci':list(ci_normal)}

### C2. Estimating both mean and width
For an unknown mean and standard deviation,
$$\hat\mu=\bar x,\quad\hat\sigma^2=\frac1n\sum_i(x_i-\bar x)^2.$$
The unbiased sample-variance estimator instead divides by $n-1$; unbiasedness and maximum likelihood are different criteria. For a joint two-parameter region, the asymptotic 68.27% likelihood-ratio cutoff is $-2\Delta\log L\simeq2.30$, not 1. For this small sample, treat that contour as an approximation. Profiling over a nuisance parameter differs from fixing it.

In [ ]:
sigma_mle=np.sqrt(np.mean((x_gauss-mu_mle)**2))
mu2=np.linspace(mu_mle-3.5,mu_mle+3.5,101)
sd2=np.linspace(0.6,5.5,101)
MU,SD=np.meshgrid(mu2,sd2)

logL2=-len(x_gauss)*np.log(SD)-0.5*np.sum((x_gauss[:,None,None]-MU)**2,axis=0)/SD**2
logL2-=(-len(x_gauss)*np.log(sigma_mle)-len(x_gauss)/2)
print('MLE sigma:',sigma_mle,'Unbiased variance estimate:',np.var(x_gauss,ddof=1),'std:',np.std(x_gauss,ddof=1))

cont=plt.contour(MU,SD,-2*logL2,levels=[1.0,2.30,5.99]);plt.clabel(cont)
plt.plot(mu_mle,sigma_mle,'rx',label='MLE')
plt.xlabel('mu');plt.ylabel('sigma');plt.legend();show()

profile_sigma=np.sqrt(np.mean((x_gauss[:,None]-mu_grid)**2,axis=0))
profile=np.array([normal_loglike(m,s) for m,s in zip(mu_grid,profile_sigma)])-normal_loglike(mu_mle,sigma_mle)
fixed=np.array([normal_loglike(m,sigma_mle) for m in mu_grid])-normal_loglike(mu_mle,sigma_mle)
record('profile',mu_grid,Profile=profile,Fixed_sigma=fixed)

### C3. Curvature errors and their limits
Near a regular interior maximum, $\log L(\theta)\simeq\log L(\hat\theta)-(\theta-\hat\theta)^2/(2s_\theta^2)$. This motivates a curvature estimate and a $\Delta\log L=-0.5$ interval for one parameter. Boundaries, few observations, skewness, non-identifiability, or wrong models can defeat this approximation.

A likelihood interval is not automatically a Bayesian credible interval. A credible interval requires a normalized posterior and a prior.

### C4. Exact binned Gaussian likelihood
For fixed total $N$, histogram counts are multinomial with bin probabilities
$$p_j(\mu)=F(e_{j+1};\mu,\sigma)-F(e_j;\mu,\sigma),\qquad\log L_{\rm bin}=\sum_j n_j\log p_j.$$
The multinomial coefficient is constant in $\mu$. Include underflow and overflow bins so probabilities sum to one. If only a selected range is observed, use a likelihood conditioned on that selection instead. A midpoint approximation $f(x_j)\Delta x_j$ can be poor for wide bins.

We compare the exact binned likelihood with the unbinned likelihood using the same sample and known $\sigma$.

In [ ]:
rng_bin=np.random.default_rng(23)
x_bin=rng_bin.normal(10,2.5,100)
coarse_edges=np.array([-np.inf,5.0,8.0,11.0,14.0,np.inf])
bin_counts=np.histogram(x_bin,coarse_edges)[0]

def bin_probabilities(mu):
    return np.diff(stats.norm.cdf(coarse_edges,loc=mu,scale=2.5))

def binned_loglike(mu):
    prob=bin_probabilities(mu)
    return float(np.sum(xlogy(bin_counts,prob)))

mu_unbin=x_bin.mean()
mu_bin=minimize_scalar(lambda m:-binned_loglike(m),bounds=(5,15),method='bounded').x
bgrid=np.linspace(8.5,11.5,401)
bscan=np.array([binned_loglike(m) for m in bgrid])-binned_loglike(mu_bin)
uscan=np.array([np.sum(stats.norm.logpdf(x_bin,m,2.5)) for m in bgrid])-np.sum(stats.norm.logpdf(x_bin,mu_unbin,2.5))
assert np.isclose(bin_probabilities(mu_bin).sum(),1)

print('Counts including tails:',bin_counts,'N:',bin_counts.sum())
print('Unbinned / binned MLE:',mu_unbin,mu_bin)

for vals,label in [(uscan,'Unbinned'),(bscan,'Exact binned')]:plt.plot(bgrid,vals,label=label)
plt.axhline(-0.5,color='gray',ls='--');plt.ylim(-10,0.2)
plt.xlabel('mu');plt.ylabel('Delta log likelihood');plt.legend();show()
record('binned',bgrid,Unbinned=uscan,Binned=bscan)

### C5. Why a histogram curve fit is a different estimator
Below, the likelihood estimate stays fixed, while the histogram-fit estimate can depend on binning. Do not infer a general ranking from one synthetic sample.

In [ ]:
rng_fit=np.random.default_rng(24)
x_fit=rng_fit.normal(10,2.5,25)
nbins_list=[5,10,20]
fit_means=[]

for bins in nbins_list:
    h,e=np.histogram(x_fit,bins=bins,density=True)
    c=(e[1:]+e[:-1])/2
    fitted,_=curve_fit(lambda xx,m:stats.norm.pdf(xx,m,2.5),c,h,p0=[x_fit.mean()])
    fit_means.append(fitted[0])

print('Unbinned MLE:',x_fit.mean())
print('Bin counts / unweighted histogram-fit means:',list(zip(nbins_list,fit_means)))

plt.plot(nbins_list,fit_means,'o-',label='Unweighted density fit')
plt.axhline(x_fit.mean(),color='black',ls='--',label='Unbinned MLE')
plt.xlabel('Number of histogram bins');plt.ylabel('Estimated mu');plt.legend();show()
record('hist_fit',nbins_list,Histogram_fit=fit_means,Unbinned_MLE=np.full(3,x_fit.mean()))

## D. Coin-flip likelihood and Bayes' rule
### D1. Coin experiment
For 450 heads in 1000 independent flips, $\hat p=k/n=0.45$ and the local standard error is $\sqrt{\hat p(1-\hat p)/n}\simeq0.0157$. Likelihood quantifies data compatibility at each $p$; it is not yet a probability distribution for $p$.

In [ ]:
n_coin,k_coin=1000,450

def coin_loglike(p):
    if not 0<p<1:return -np.inf
    return k_coin*np.log(p)+(n_coin-k_coin)*np.log1p(-p)

p_mle=k_coin/n_coin
ci_coin=interval_from_loglike(coin_loglike,p_mle,0.01,0.99)
pgrid=np.linspace(0.37,0.53,401)
coin_scan=np.array([coin_loglike(p) for p in pgrid])-coin_loglike(p_mle)

print('p MLE:',p_mle,'Local SE:',np.sqrt(p_mle*(1-p_mle)/n_coin))
print('Delta-logL = -0.5 interval:',ci_coin)

plt.plot(pgrid,coin_scan);plt.axhline(-0.5,color='gray',ls='--');plt.ylim(-12,0.2)
plt.xlabel('Head probability p');plt.ylabel('Delta log likelihood');show()
record('coin_likelihood',pgrid,Delta_logL=coin_scan)

### D2. Bayes' rule for discrete hypotheses
$$P(H_j\mid D)=\frac{P(D\mid H_j)P(H_j)}{\sum_kP(D\mid H_k)P(H_k)}.$$
The alternatives in the denominator must be mutually exclusive and exhaustive.

**Two boxes:** equal prior probabilities, with blue fractions 0.01 in A and 0.99 in B.  
**Synthetic diagnostic example:** prevalence 0.01, sensitivity 0.99, false-positive probability 0.10. These are teaching assumptions from the slides, not properties of any actual medical test.

In [ ]:
def discrete_update(prior,likelihood):
    joint=np.asarray(prior)*np.asarray(likelihood)
    evidence=joint.sum()
    if evidence<=0:raise ValueError('Observed event has zero predictive probability.')
    return joint/evidence,evidence

box_post,box_evidence=discrete_update([0.5,0.5],[0.01,0.99])
test_post,test_evidence=discrete_update([0.01,0.99],[0.99,0.10])

print('P(A|blue):',box_post[0])
print('P(condition|positive), synthetic example:',test_post[0])
print('Per 10,000 people: expected true positives 99, false positives 990.')
assert np.allclose(box_post,[0.01,0.99])
assert np.isclose(test_post[0],1/11)

prevalence=np.linspace(0.001,0.20,200)
ppv=0.99*prevalence/(0.99*prevalence+0.10*(1-prevalence))
plt.plot(prevalence,ppv);plt.xlabel('Prior prevalence');plt.ylabel('P(condition | positive)');show()
record('base_rate',prevalence,Posterior=ppv)

### D3. A short bridge to continuous Bayesian analysis
Combining the binomial likelihood with $p\sim\mathrm{Beta}(a,b)$ yields $p\mid D\sim\mathrm{Beta}(a+k,b+n-k)$. Under a uniform prior, the 450/1000 experiment gives Beta(451,551).

This exact example connects the prerequisite material to the later Bayesian-analysis course. The advanced topics (MCMC, GP emulation, and PCA) are intentionally left to that separate lecture. A continuous posterior assigns zero probability to the exact point $p=0.5$; testing an exactly fair coin as a separate model requires a point mass and model prior.

In [ ]:
coin_post=stats.beta(451,551)
print('Posterior mean:',coin_post.mean())
print('95% equal-tailed credible interval:',coin_post.interval(0.95))
print('P(p<0.5|data):',coin_post.cdf(0.5))
plt.plot(pgrid,coin_post.pdf(pgrid));plt.xlabel('p');plt.ylabel('Posterior density');show()
record('coin_posterior',pgrid,Posterior=coin_post.pdf(pgrid))

## E. Detector response and iterative Bayesian unfolding
### E1. Response model
The supplied example uses a Breit–Wigner (Cauchy) shape with center 0.3, full width 2.5, detector shift −2, and Gaussian resolution 1:
$$x_{\rm meas}=x_{\rm true}-2+\epsilon,\quad \epsilon\sim\mathcal N(0,1).$$
The Cauchy distribution has no finite mean or variance; describe its location and width rather than interpreting a sample mean as its true mean.

Our binned teaching version uses 30 bins over [−15,15]. The truth population is explicitly restricted to this range, and truth is approximated as uniform within each bin. This makes the response definition precise and avoids unmodeled feed-in from outside the truth domain. Bin masses follow the truncated Cauchy CDF. Measured events can be lost outside the acceptance.

A response column is $R_{ij}=P(i_{\rm meas}\mid j_{\rm true})$, and its sum is the efficiency $\epsilon_j\le1$. We retain losses rather than normalizing each accepted column to one.

In [ ]:
edges_resp=np.linspace(-15,15,31)
x_centers=0.5*(edges_resp[1:]+edges_resp[:-1])
n_bins=len(x_centers)
truth_prob=np.diff(stats.cauchy.cdf(edges_resp,loc=0.3,scale=2.5/2))
truth_prob/=truth_prob.sum()
truth_expected=100*truth_prob
shift,resolution=-2.0,1.0

# Integrate the response over a uniform distribution within each truth bin.
quad_x,quad_w=np.polynomial.legendre.leggauss(40)
R_exact=np.empty((n_bins,n_bins))

for j in range(n_bins):
    points=(edges_resp[j]+edges_resp[j+1])/2+quad_x*(edges_resp[j+1]-edges_resp[j])/2
    cdf=stats.norm.cdf((edges_resp[:,None]-(points[None,:]+shift))/resolution)
    R_exact[:,j]=np.diff(cdf,axis=0)@quad_w/2

mean_measured=R_exact@truth_expected
measured=np.random.default_rng(29).poisson(mean_measured)

print('Expected truth / accepted mean / observed total:',truth_expected.sum(),mean_measured.sum(),measured.sum())
print('Efficiency range:',R_exact.sum(axis=0).min(),R_exact.sum(axis=0).max())
assert np.all(R_exact>=0) and np.all(R_exact.sum(axis=0)<=1+1e-12)

fig,ax=plt.subplots(1,2,figsize=(11,4))
ax[0].plot(x_centers,truth_expected,label='Truth expectation');ax[0].plot(x_centers,mean_measured,label='Measured expectation');ax[0].step(x_centers,measured,where='mid',label='Observed counts');ax[0].legend(fontsize=8);ax[0].set(xlabel='x',ylabel='Counts per bin')
im=ax[1].imshow(R_exact,origin='lower',extent=[-15,15,-15,15],aspect='auto');ax[1].set(xlabel='Truth x',ylabel='Measured x',title='Response probability');fig.colorbar(im,ax=ax[1]);show()
record('smearing',x_centers,Truth=truth_expected,Measured_mean=mean_measured,Observed=measured)
record('response_columns',x_centers,Left=R_exact[:,1],Center=R_exact[:,15],Right=R_exact[:,28])
record('efficiency',x_centers,Efficiency=R_exact.sum(axis=0))

### E2. Finite Monte Carlo response statistics
Build each column from independently generated detector simulations. Divide migration counts by **all generated events in that truth bin**, including those lost outside measured acceptance. The column sum then estimates efficiency.

We compare 1,000 and 100,000 total MC events, stratified across truth bins. Stratification supplies coverage of every truth bin, but migration cells can still be empty. A likelihood should not silently discard observed bins that the estimated model considers impossible.

In [ ]:
def simulate_response(total_mc,seed):
    rng=np.random.default_rng(seed)
    R=np.zeros_like(R_exact)
    generated=np.zeros(n_bins,dtype=int)
    for j in range(n_bins):
        nj=total_mc//n_bins+int(j<total_mc%n_bins)
        generated[j]=nj
        xt=rng.uniform(edges_resp[j],edges_resp[j+1],nj)
        xm=xt+shift+rng.normal(0,resolution,nj)
        R[:,j]=np.histogram(xm,bins=edges_resp)[0]/nj
    return R,generated

R_low,gen_low=simulate_response(1000,301)
R_high,gen_high=simulate_response(100000,302)
assert gen_low.sum()==1000 and gen_high.sum()==100000

print('Response RMS error, 1k / 100k MC:',np.sqrt(np.mean((R_low-R_exact)**2)),np.sqrt(np.mean((R_high-R_exact)**2)))
plt.plot(x_centers,R_exact[:,15],label='Integrated response')
plt.plot(x_centers,R_low[:,15],'o-',label='1,000 MC total')
plt.plot(x_centers,R_high[:,15],'.-',label='100,000 MC total')
plt.xlabel('Measured x');plt.ylabel('P(measured bin | central truth bin)');plt.legend();show()
record('mc_response',x_centers,Integrated=R_exact[:,15],MC_1000=R_low[:,15],MC_100000=R_high[:,15])

### E3. Bayes' rule inside unfolding
For a prior truth shape $\pi_j$,
$$P(j\mid i)=\frac{R_{ij}\pi_j}{\sum_kR_{ik}\pi_k}.$$
In a count-based iteration, the equivalent update is
$$t_j^{(r+1)}=\frac{t_j^{(r)}}{\epsilon_j}\sum_i R_{ij}\frac{n_i}{(Rt^{(r)})_i}.$$
This is for a known response, no background, and positive efficiency. When efficiency is one, it reduces to the normalized-source construction after accounting for the total count. Repeated use of the same data is an iterative estimation algorithm, not multiplication by a new independent likelihood at each step.

There is no division by observed counts. Empty measured bins are valid. An observed bin with positive count and zero predicted support is a model failure that needs investigation.

In [ ]:
def unfold(counts,R,iterations=10,start=None):
    counts=np.asarray(counts,dtype=float)
    efficiency=R.sum(axis=0)
    if np.any(efficiency<=0):raise ValueError('Unidentifiable truth bin: zero efficiency.')
    t=np.full(R.shape[1],max(counts.sum(),1)/np.sum(efficiency)) if start is None else np.asarray(start,dtype=float).copy()
    if np.any(t<=0):raise ValueError('Use a strictly positive starting spectrum.')
    history=[]
    for _ in range(iterations):
        predicted=R@t
        if np.any((predicted==0)&(counts>0)):
            raise ValueError('Observed counts lie outside the response support.')
        ratio=np.divide(counts,predicted,out=np.zeros_like(counts),where=predicted>0)
        t=t*(R.T@ratio)/efficiency
        history.append(t.copy())
    return np.asarray(history)

history=unfold(measured,R_high,10)
low_history=unfold(measured,R_low,10)

for iteration in [1,2,4,10]:plt.plot(x_centers,history[iteration-1],label=f'Iteration {iteration}')

plt.plot(x_centers,truth_expected,'k--',label='Generating expectation')
plt.xlim(-8,8);plt.xlabel('Truth x');plt.ylabel('Estimated truth counts');plt.legend();show()
print('Unfolded totals at 1, 4, 10 iterations:',history[[0,3,9]].sum(axis=1))
record('iterations',x_centers,Truth=truth_expected,Iteration_1=history[0],Iteration_4=history[3],Iteration_10=history[9])

### E4. Closure, regularization, and finite MC
First test on an Asimov sample (counts equal expected means) with the same known response. This checks the algorithm without one particular statistical fluctuation. It is not an independent validation of the detector model.

For real data, compare refolded predictions $R\hat t$ with measured counts. Small refolding residuals do not imply a unique or accurate truth spectrum. Increasing iterations tends to fit more sampling noise. Select iteration count through an explicit simulation study, not a visual preference on the observed sample.

In [ ]:
asimov_history=unfold(mean_measured,R_exact,100)
initial=np.full(n_bins,mean_measured.sum()/R_exact.sum())
initial_error=np.linalg.norm(initial-truth_expected)
closure_error=np.linalg.norm(asimov_history[-1]-truth_expected)
assert closure_error<initial_error

identity_counts=np.array([2.0,0.0,4.0])
assert np.allclose(unfold(identity_counts,np.eye(3),1)[0],identity_counts)
refolded=R_high@history[3]

print('Asimov initial / 100-iteration truth error:',initial_error,closure_error)
print('At 4 iterations, response-MC comparison:')
print('1k-MC / 100k-MC unfolded total:',low_history[3].sum(),history[3].sum())

plt.step(x_centers,measured,where='mid',label='Observed')
plt.plot(x_centers,refolded,label='Refolded, iteration 4')
plt.xlabel('Measured x');plt.ylabel('Counts');plt.legend();show()
record('refolding',x_centers,Observed=measured,Refolded=refolded)
record('mc_unfold',x_centers,Truth=truth_expected,MC_1000=low_history[3],MC_100000=history[3])

## F. Uncertainty and a bridge to full Bayesian inference
### F1. Parametric bootstrap conditional on a fixed response
We generate Poisson pseudo-experiments around the refolded fitted expectation and repeat the same four-iteration estimator. The resulting covariance describes sampling variation **conditional on the fitted response and this procedure**. It is not a Bayesian posterior, does not include finite-MC response uncertainty, and does not remove regularization bias.

A fuller uncertainty study would also fluctuate or model the response, vary relevant nuisance parameters, and check coverage using independent synthetic truths.

In [ ]:
rng_boot=np.random.default_rng(401)
bootstrap=np.array([unfold(rng_boot.poisson(refolded),R_high,4)[-1] for _ in range(300)])
cov_unfold=np.cov(bootstrap,rowvar=False)
sd_unfold=np.sqrt(np.diag(cov_unfold))

plt.errorbar(x_centers,history[3],yerr=sd_unfold,fmt='o',label='4 iterations, bootstrap SD')
plt.plot(x_centers,truth_expected,'k--',label='Generating expectation')
plt.xlim(-8,8);plt.xlabel('Truth x');plt.ylabel('Counts');plt.legend();show()

print('Adjacent-bin covariance near center:',cov_unfold[14,15])
assert np.min(np.linalg.eigvalsh(cov_unfold))>-1e-10
record('bootstrap',x_centers,Estimate=history[3],Lower=history[3]-sd_unfold,Upper=history[3]+sd_unfold)

### F2. Forward folding as an alternative inverse problem
Instead of estimating every truth bin, assume a parameterized spectrum $t_j(\theta)$ and fit measured counts with
$$n_i\sim\mathrm{Poisson}\left(\sum_jR_{ij}t_j(\theta)+b_i\right).$$
Below the width is known, background is zero, and we estimate the Cauchy location and total truth yield. It uses the same binned truth-domain and within-bin-uniform assumptions as E1. A Bayesian extension multiplies this likelihood by priors for the location and yield, then normalizes or samples.

In [ ]:
def expected_truth(location,total):
    masses=np.diff(stats.cauchy.cdf(edges_resp,loc=location,scale=2.5/2))
    return total*masses/masses.sum()

def forward_nll(params):
    location,log_total=params
    prediction=R_high@expected_truth(location,np.exp(log_total))
    return -np.sum(stats.poisson.logpmf(measured,prediction))

fit=minimize(forward_nll,[0.0,np.log(max(measured.sum(),1))],method='L-BFGS-B',bounds=[(-3,3),(np.log(1),np.log(1000))])
assert fit.success
fit_location,fit_total=float(fit.x[0]),float(np.exp(fit.x[1]))
fit_truth=expected_truth(fit_location,fit_total)

print('Forward-folded location MLE:',fit_location,'Total truth-yield MLE:',fit_total)
print('Generating location / expected truth yield:',0.3,100)

plt.step(x_centers,measured,where='mid',label='Observed counts')
plt.plot(x_centers,R_high@fit_truth,label='Forward-folded fit')
plt.xlabel('Measured x');plt.ylabel('Counts');plt.legend();show()
record('forward_fit',x_centers,Observed=measured,Fitted=R_high@fit_truth)
slide_data['forward_summary']={'location':fit_location,'yield':fit_total}